# Notebook 10 — Cross-window robustness aggregation (fixed + expanded)

This version is hardened for the crash-resilient multiwindow workflow.

What it fixes:
- works with `multiwindow_manifest.csv` rows from the crash-resilient notebook
- handles `requested_start` / `requested_end` manifest columns instead of assuming `start` / `end`
- filters to windows with `window_validation_status == "ok"` by default
- finds output CSVs **recursively**, so nested paths like `regimes/regimes/regime_labels.csv` are supported
- avoids treating audit files as science outputs
- normalizes common column-name variations before aggregation

What it adds:
- long-lead and **leadwise** AI-vs-HRES delta summaries
- model win-share tables across windows and leads
- regime-balance **and entropy** summaries
- blocking-penalty summaries and blocking event-skill summaries
- a file-inventory table for debugging provenance


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def detect_bundle_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        Path("/global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass"),
    ]
    for base in candidates:
        if (base / "src" / "flagship_predictability").exists() and (base / "examples" / "default_settings.py").exists():
            return base
        alt = base / "flagship_predictability_nextpass"
        if (alt / "src" / "flagship_predictability").exists() and (alt / "examples" / "default_settings.py").exists():
            return alt
    raise RuntimeError("Could not detect bundle root; edit detect_bundle_root() with the correct project path.")

BUNDLE_ROOT = detect_bundle_root()
SRC_ROOT = BUNDLE_ROOT / "src"
NOTEBOOK_ROOT = BUNDLE_ROOT / "notebooks"
OUTPUT_ROOT = NOTEBOOK_ROOT / "outputs" / "flagship_96plus"
MULTIWINDOW_ROOT = OUTPUT_ROOT / "multiwindow_runs"
DENSE_ROOT = OUTPUT_ROOT / "denselead_upgrade"
ROBUST_ROOT = OUTPUT_ROOT / "robustness_tables"
ROBUST_ROOT.mkdir(parents=True, exist_ok=True)

for p in [BUNDLE_ROOT, SRC_ROOT]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("Bundle root:", BUNDLE_ROOT)
print("Multiwindow root:", MULTIWINDOW_ROOT)
print("Robustness root:", ROBUST_ROOT)


Bundle root: /global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass
Multiwindow root: /global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_96plus/multiwindow_runs
Robustness root: /global/u2/s/suryact/Chaos_Theory/project1_code/flagship_predictability_nextpass/notebooks/outputs/flagship_96plus/robustness_tables


In [2]:
PREFER_VALIDATED_OK = True
DROP_DUPLICATE_TAGS_KEEP_LAST = True

def safe_read_csv(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"Could not read {path}: {e}")
        return pd.DataFrame()

def coalesce(*vals):
    for v in vals:
        if pd.isna(v) if isinstance(v, float) else False:
            continue
        if v is not None:
            return v
    return None

def bootstrap_ci(values, n_boot=2000, alpha=0.05, seed=42):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan, np.nan, np.nan
    if vals.size == 1:
        x = float(vals[0])
        return x, x, x
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot, dtype=float)
    n = vals.size
    for i in range(n_boot):
        means[i] = rng.choice(vals, size=n, replace=True).mean()
    lo, hi = np.quantile(means, [alpha/2, 1-alpha/2])
    return float(vals.mean()), float(lo), float(hi)

def parse_lead_hours_series(series):
    s = series.copy()
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").astype(float)
    out = pd.Series(np.nan, index=s.index, dtype=float)
    # try timedelta-like first
    try:
        td = pd.to_timedelta(s, errors="coerce")
        mask = td.notna()
        if mask.any():
            out.loc[mask] = td.loc[mask].dt.total_seconds() / 3600.0
    except Exception:
        pass
    # plain hour strings / ints
    mask = out.isna()
    if mask.any():
        tmp = s.astype(str).str.extract(r"(-?\d+(?:\.\d+)?)")[0]
        out.loc[mask] = pd.to_numeric(tmp.loc[mask], errors="coerce")
    return out

def season_order_key(x):
    order = {"DJF": 0, "MAM": 1, "JJA": 2, "SON": 3}
    return order.get(str(x), 99)

def infer_bucket_from_tag(tag):
    tag = str(tag)
    if tag == "all_2018_2022" or tag.startswith("all_"):
        return "all_years"
    if tag.isdigit():
        return "year"
    if re.match(r"^(DJF|MAM|JJA|SON)_\d{4}$", tag):
        return "season"
    return None

def infer_season_from_tag(tag):
    tag = str(tag)
    m = re.match(r"^(DJF|MAM|JJA|SON)_\d{4}$", tag)
    return m.group(1) if m else None

def normalize_model_name(x):
    x = str(x)
    lx = x.lower()
    if "graphcast" in lx:
        return "GraphCast"
    if "hres" in lx:
        return "HRES"
    if "neuralgcm" in lx:
        return "NeuralGCM"
    return x

def find_candidate_files(base, basename):
    base = Path(base)
    if not base.exists():
        return []
    return sorted(base.rglob(basename))

def choose_best_file(candidates, basename):
    if not candidates:
        return None
    scored = []
    for p in candidates:
        parts = [x.lower() for x in p.parts]
        score = 0
        if basename in p.name:
            score += 5
        if "audit" in parts:
            score -= 10
        if "validation" in parts:
            score -= 3
        if "regimes" in parts and basename == "regime_labels.csv":
            score += 8
        if "deterministic" in parts and basename in {"deterministic_summary.csv", "regime_conditioned_metrics.csv"}:
            score += 8
        if "blocking" in parts and basename in {"blocking_rmse.csv", "blocking_event_metrics.csv"}:
            score += 8
        score -= len(p.parts) * 0.01  # very mild preference for shallower paths
        try:
            score += min(p.stat().st_size / 1e6, 5.0)
        except Exception:
            pass
        scored.append((score, p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]

def normalize_manifest(manifest):
    m = manifest.copy()
    if DROP_DUPLICATE_TAGS_KEEP_LAST and "tag" in m.columns:
        m = m.drop_duplicates(subset=["tag"], keep="last").reset_index(drop=True)

    if "bucket" not in m.columns:
        m["bucket"] = m["tag"].map(infer_bucket_from_tag)
    else:
        m["bucket"] = m["bucket"].fillna(m["tag"].map(infer_bucket_from_tag))
    if "season" not in m.columns:
        m["season"] = m["tag"].map(infer_season_from_tag)
    else:
        m["season"] = m["season"].fillna(m["tag"].map(infer_season_from_tag))

    if "start" not in m.columns:
        m["start"] = None
    if "end" not in m.columns:
        m["end"] = None
    if "requested_start" in m.columns:
        m["start"] = m["start"].fillna(m["requested_start"])
    if "requested_end" in m.columns:
        m["end"] = m["end"].fillna(m["requested_end"])

    if "run_root" not in m.columns:
        m["run_root"] = m["tag"].map(lambda t: str(MULTIWINDOW_ROOT / str(t)))
    else:
        m["run_root"] = m["run_root"].fillna(m["tag"].map(lambda t: str(MULTIWINDOW_ROOT / str(t))))

    if PREFER_VALIDATED_OK and "window_validation_status" in m.columns:
        m = m[m["window_validation_status"].astype(str).str.lower() == "ok"].copy()

    if "worker_process_status" in m.columns:
        ok_mask = m["worker_process_status"].astype(str).str.lower().isin(["ok", "nan"])
        if ok_mask.any():
            m = m[ok_mask].copy()

    return m.reset_index(drop=True)

def normalize_det(df):
    if df.empty:
        return df
    out = df.copy()
    rename = {
        "var": "variable",
        "forecast_model": "model",
        "rmse": "rmse_mean",
        "acc": "acc_mean",
        "mean_rmse": "rmse_mean",
        "mean_acc": "acc_mean",
        "horizon": "lead",
        "lead_time": "lead",
        "lead_td": "lead",
    }
    out = out.rename(columns={k:v for k,v in rename.items() if k in out.columns and v not in out.columns})
    if "model" in out.columns:
        out["model"] = out["model"].map(normalize_model_name)
    if "lead_h" not in out.columns and "lead" in out.columns:
        out["lead_h"] = parse_lead_hours_series(out["lead"])
    return out

def normalize_reg(df):
    if df.empty:
        return df
    out = df.copy()
    rename = {"var":"variable", "forecast_model":"model", "value":"mean", "lead_time":"lead", "lead_td":"lead"}
    out = out.rename(columns={k:v for k,v in rename.items() if k in out.columns and v not in out.columns})
    if "model" in out.columns:
        out["model"] = out["model"].map(normalize_model_name)
    if "lead_h" not in out.columns and "lead" in out.columns:
        out["lead_h"] = parse_lead_hours_series(out["lead"])
    return out

def normalize_blk(df):
    if df.empty:
        return df
    out = df.copy()
    rename = {"forecast_model":"model", "lead_time":"lead", "lead_td":"lead"}
    out = out.rename(columns={k:v for k,v in rename.items() if k in out.columns and v not in out.columns})
    if "model" in out.columns:
        out["model"] = out["model"].map(normalize_model_name)
    if "lead_h" not in out.columns and "lead" in out.columns:
        out["lead_h"] = parse_lead_hours_series(out["lead"])
    return out

def normalize_evt(df):
    if df.empty:
        return df
    out = df.copy()
    rename = {"forecast_model":"model", "lead_time":"lead", "lead_td":"lead"}
    out = out.rename(columns={k:v for k,v in rename.items() if k in out.columns and v not in out.columns})
    if "model" in out.columns:
        out["model"] = out["model"].map(normalize_model_name)
    if "lead_h" not in out.columns and "lead" in out.columns:
        out["lead_h"] = parse_lead_hours_series(out["lead"])
    return out

def normalize_labels(df):
    if df.empty:
        return df
    out = df.copy()
    if "time" in out.columns:
        out["time"] = pd.to_datetime(out["time"], errors="coerce")
    if "regime" not in out.columns:
        for c in out.columns:
            if c.lower().startswith("regime"):
                out = out.rename(columns={c:"regime"})
                break
    return out


In [3]:
import re

manifest_path = MULTIWINDOW_ROOT / "multiwindow_manifest.csv"
if not manifest_path.exists():
    raise FileNotFoundError(f"Could not find {manifest_path}. Run Notebook 08 first.")

manifest_raw = pd.read_csv(manifest_path)
manifest = normalize_manifest(manifest_raw)

validation_path = MULTIWINDOW_ROOT / "multiwindow_validation.csv"
validation_df = safe_read_csv(validation_path)
if not validation_df.empty and "tag" in validation_df.columns:
    validation_df = validation_df.drop_duplicates(subset=["tag"], keep="last")

coverage = manifest.groupby(["bucket", "season"], dropna=False).size().reset_index(name="n_windows")
coverage.to_csv(ROBUST_ROOT / "window_coverage_summary.csv", index=False)
manifest.to_csv(ROBUST_ROOT / "eligible_multiwindow_manifest.csv", index=False)

print("Raw manifest rows:", len(manifest_raw))
print("Eligible manifest rows:", len(manifest))
display(coverage)
display(manifest.head())
if not validation_df.empty:
    display(validation_df.head())


Raw manifest rows: 25
Eligible manifest rows: 24


,bucket,season,n_windows
0,season,DJF,4
1,season,JJA,5
2,season,MAM,5
3,season,SON,5
4,year,ALL,5


,tag,bucket,season,requested_start,requested_end,run_root,prerun_date_windows_contains_requested,prerun_observed_date_windows,audit_status,regimes_status,...,growth_status,blocking_status,regime_sensitivity_status,subprocess_returncode,elapsed_seconds,stdout_log,stderr_log,worker_process_status,start,end
0,MAM_2018,season,MAM,2018-03-01,2018-05-31,/global/u2/s/suryact/Chaos_Theory/project1_cod...,True,"[[""2018-03-01"", ""2018-05-31""]]",ok,ok,...,ok,ok,skipped,0.0,355.29,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,ok,2018-03-01,2018-05-31
1,JJA_2018,season,JJA,2018-06-01,2018-08-31,/global/u2/s/suryact/Chaos_Theory/project1_cod...,True,"[[""2018-06-01"", ""2018-08-31""]]",ok,ok,...,ok,ok,skipped,0.0,363.76,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,ok,2018-06-01,2018-08-31
2,SON_2018,season,SON,2018-09-01,2018-11-30,/global/u2/s/suryact/Chaos_Theory/project1_cod...,True,"[[""2018-09-01"", ""2018-11-30""]]",ok,ok,...,ok,ok,skipped,0.0,331.09,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,ok,2018-09-01,2018-11-30
3,DJF_2019,season,DJF,2018-12-01,2019-02-28,/global/u2/s/suryact/Chaos_Theory/project1_cod...,True,"[[""2018-12-01"", ""2019-02-28""]]",ok,ok,...,ok,ok,skipped,0.0,336.19,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,ok,2018-12-01,2019-02-28
4,MAM_2019,season,MAM,2019-03-01,2019-05-31,/global/u2/s/suryact/Chaos_Theory/project1_cod...,True,"[[""2019-03-01"", ""2019-05-31""]]",ok,ok,...,ok,ok,skipped,0.0,341.65,/global/u2/s/suryact/Chaos_Theory/project1_cod...,/global/u2/s/suryact/Chaos_Theory/project1_cod...,ok,2019-03-01,2019-05-31


,tag,requested_start,requested_end,primary_file,primary_time_col,primary_observed_start,primary_observed_end,primary_time_hash,primary_n_unique_times,validation_status
0,2018,2018-01-01 00:00:00,2018-12-31 00:00:00,/global/u2/s/suryact/Chaos_Theory/project1_cod...,time,2018-01-01 00:00:00,2018-12-31 18:00:00,a2591f0adee7bd8abf763c78d39090c4,1460,ok
1,2019,2019-01-01 00:00:00,2019-12-31 00:00:00,/global/u2/s/suryact/Chaos_Theory/project1_cod...,time,2019-01-01 00:00:00,2019-12-31 18:00:00,d97d831bf9b5aac258e9d1e4ddc184bc,1460,ok
2,2020,2020-01-01 00:00:00,2020-12-31 00:00:00,/global/u2/s/suryact/Chaos_Theory/project1_cod...,time,2020-01-01 00:00:00,2020-12-31 18:00:00,84ef922b0dfb82f9fe10e7797be5d84b,1464,ok
3,2021,2021-01-01 00:00:00,2021-12-31 00:00:00,/global/u2/s/suryact/Chaos_Theory/project1_cod...,time,2021-01-01 00:00:00,2021-12-31 18:00:00,f85a8afdf7493008e7ae144aebce859e,1460,ok
4,2022,2022-01-01 00:00:00,2022-12-31 00:00:00,/global/u2/s/suryact/Chaos_Theory/project1_cod...,time,2022-01-01 00:00:00,2022-12-31 18:00:00,abcb094f9c440018919d47e8709b8e36,1460,ok


In [4]:
BASENAMES = [
    "deterministic_summary.csv",
    "regime_conditioned_metrics.csv",
    "blocking_rmse.csv",
    "blocking_event_metrics.csv",
    "regime_labels.csv",
]

def collect_outputs(manifest_df):
    det_rows, reg_rows, blk_rows, evt_rows, lab_rows, inventory_rows = [], [], [], [], [], []

    for _, row in manifest_df.iterrows():
        base = Path(row["run_root"])
        meta = {
            "tag": row.get("tag"),
            "bucket": row.get("bucket"),
            "season": row.get("season"),
            "start": row.get("start"),
            "end": row.get("end"),
        }

        chosen = {}
        for bn in BASENAMES:
            chosen[bn] = choose_best_file(find_candidate_files(base, bn), bn)
            inventory_rows.append({
                **meta,
                "basename": bn,
                "chosen_file": str(chosen[bn]) if chosen[bn] is not None else None,
                "found_count": len(find_candidate_files(base, bn)),
            })

        det = normalize_det(safe_read_csv(chosen["deterministic_summary.csv"])) if chosen["deterministic_summary.csv"] else pd.DataFrame()
        reg = normalize_reg(safe_read_csv(chosen["regime_conditioned_metrics.csv"])) if chosen["regime_conditioned_metrics.csv"] else pd.DataFrame()
        blk = normalize_blk(safe_read_csv(chosen["blocking_rmse.csv"])) if chosen["blocking_rmse.csv"] else pd.DataFrame()
        evt = normalize_evt(safe_read_csv(chosen["blocking_event_metrics.csv"])) if chosen["blocking_event_metrics.csv"] else pd.DataFrame()
        lab = normalize_labels(safe_read_csv(chosen["regime_labels.csv"])) if chosen["regime_labels.csv"] else pd.DataFrame()

        for df in [det, reg, blk, evt, lab]:
            if not df.empty:
                for k, v in meta.items():
                    df[k] = v

        if not det.empty: det_rows.append(det)
        if not reg.empty: reg_rows.append(reg)
        if not blk.empty: blk_rows.append(blk)
        if not evt.empty: evt_rows.append(evt)
        if not lab.empty: lab_rows.append(lab)

    inventory = pd.DataFrame(inventory_rows)
    return (
        pd.concat(det_rows, ignore_index=True) if det_rows else pd.DataFrame(),
        pd.concat(reg_rows, ignore_index=True) if reg_rows else pd.DataFrame(),
        pd.concat(blk_rows, ignore_index=True) if blk_rows else pd.DataFrame(),
        pd.concat(evt_rows, ignore_index=True) if evt_rows else pd.DataFrame(),
        pd.concat(lab_rows, ignore_index=True) if lab_rows else pd.DataFrame(),
        inventory,
    )

det_all, reg_all, blk_all, evt_all, lab_all, file_inventory = collect_outputs(manifest)

file_inventory.to_csv(ROBUST_ROOT / "collected_file_inventory.csv", index=False)
for name, df in [("det_all", det_all), ("reg_all", reg_all), ("blk_all", blk_all), ("evt_all", evt_all), ("lab_all", lab_all)]:
    print(name, df.shape)
display(file_inventory.head(20))


det_all (1344, 15)
reg_all (11224, 18)
blk_all (288, 13)
evt_all (576, 20)
lab_all (14248, 8)


,tag,bucket,season,start,end,basename,chosen_file,found_count
0,MAM_2018,season,MAM,2018-03-01,2018-05-31,deterministic_summary.csv,/global/u2/s/suryact/Chaos_Theory/project1_cod...,1
1,MAM_2018,season,MAM,2018-03-01,2018-05-31,regime_conditioned_metrics.csv,/global/u2/s/suryact/Chaos_Theory/project1_cod...,1
2,MAM_2018,season,MAM,2018-03-01,2018-05-31,blocking_rmse.csv,/global/u2/s/suryact/Chaos_Theory/project1_cod...,1
3,MAM_2018,season,MAM,2018-03-01,2018-05-31,blocking_event_metrics.csv,/global/u2/s/suryact/Chaos_Theory/project1_cod...,1
4,MAM_2018,season,MAM,2018-03-01,2018-05-31,regime_labels.csv,/global/u2/s/suryact/Chaos_Theory/project1_cod...,1
5,JJA_2018,season,JJA,2018-06-01,2018-08-31,deterministic_summary.csv,/global/u2/s/suryact/Chaos_Theory/project1_cod...,1
6,JJA_2018,season,JJA,2018-06-01,2018-08-31,regime_conditioned_metrics.csv,/global/u2/s/suryact/Chaos_Theory/project1_cod...,1
7,JJA_2018,season,JJA,2018-06-01,2018-08-31,blocking_rmse.csv,/global/u2/s/suryact/Chaos_Theory/project1_cod...,1
8,JJA_2018,season,JJA,2018-06-01,2018-08-31,blocking_event_metrics.csv,/global/u2/s/suryact/Chaos_Theory/project1_cod...,1
9,JJA_2018,season,JJA,2018-06-01,2018-08-31,regime_labels.csv,/global/u2/s/suryact/Chaos_Theory/project1_cod...,1


In [5]:
# Deterministic aggregation and pairwise deltas

rank_stability = pd.DataFrame()
pairwise_by_window = pd.DataFrame()
pairwise_summary = pd.DataFrame()
leadwise_pairwise_by_window = pd.DataFrame()
leadwise_pairwise_summary = pd.DataFrame()
winshare_by_lead = pd.DataFrame()

if not det_all.empty and {"tag", "bucket", "season", "variable", "model"}.issubset(det_all.columns):
    work = det_all.copy()
    if "lead_h" not in work.columns and "lead" in work.columns:
        work["lead_h"] = parse_lead_hours_series(work["lead"])
    work["lead_h"] = pd.to_numeric(work["lead_h"], errors="coerce")
    work = work[work["lead_h"].notna()].copy()

    long = work[work["lead_h"] == work.groupby(["tag", "variable"])["lead_h"].transform("max")].copy()

    if {"rmse_mean", "acc_mean"}.issubset(long.columns):
        rank_stability = long.sort_values(["tag", "variable", "rmse_mean"]).copy()
        rank_stability["rank_rmse"] = rank_stability.groupby(["tag", "variable"])["rmse_mean"].rank(method="first")
        rank_stability["rank_acc"] = rank_stability.groupby(["tag", "variable"])["acc_mean"].rank(method="first", ascending=False)
        rank_stability.to_csv(ROBUST_ROOT / "longlead_rank_stability.csv", index=False)

    def model_col(df, key):
        key = key.lower()
        for c in df.columns:
            if key in str(c).lower():
                return c
        return None

    rows_long, rows_leadwise = [], []
    for metric in [m for m in ["rmse_mean", "acc_mean"] if m in work.columns]:
        # leadwise deltas
        piv_all = work.pivot_table(index=["tag", "bucket", "season", "variable", "lead_h"], columns="model", values=metric)
        hres = model_col(piv_all, "hres")
        gc = model_col(piv_all, "graphcast")
        ng = model_col(piv_all, "neuralgcm")
        for label, col in [("GraphCast_minus_HRES", gc), ("NeuralGCM_minus_HRES", ng)]:
            if hres is None or col is None:
                continue
            delta = piv_all[col] - piv_all[hres]
            tmp = delta.reset_index(name="delta")
            tmp["pair"] = label
            tmp["metric"] = metric
            tmp["favorable"] = (tmp["delta"] < 0) if metric == "rmse_mean" else (tmp["delta"] > 0)
            rows_leadwise.append(tmp)

        # long-lead deltas
        if not long.empty and metric in long.columns:
            piv_long = long.pivot_table(index=["tag", "bucket", "season", "variable"], columns="model", values=metric)
            hres = model_col(piv_long, "hres")
            gc = model_col(piv_long, "graphcast")
            ng = model_col(piv_long, "neuralgcm")
            for label, col in [("GraphCast_minus_HRES", gc), ("NeuralGCM_minus_HRES", ng)]:
                if hres is None or col is None:
                    continue
                delta = piv_long[col] - piv_long[hres]
                tmp = delta.reset_index(name="delta")
                tmp["pair"] = label
                tmp["metric"] = metric
                tmp["favorable"] = (tmp["delta"] < 0) if metric == "rmse_mean" else (tmp["delta"] > 0)
                rows_long.append(tmp)

    pairwise_by_window = pd.concat(rows_long, ignore_index=True) if rows_long else pd.DataFrame()
    pairwise_by_window.to_csv(ROBUST_ROOT / "pairwise_longlead_deltas_by_window.csv", index=False)

    leadwise_pairwise_by_window = pd.concat(rows_leadwise, ignore_index=True) if rows_leadwise else pd.DataFrame()
    leadwise_pairwise_by_window.to_csv(ROBUST_ROOT / "pairwise_leadwise_deltas_by_window.csv", index=False)

    agg_rows = []
    if not pairwise_by_window.empty:
        for keys, grp in pairwise_by_window.groupby(["bucket", "season", "variable", "pair", "metric"], dropna=False):
            mean, lo, hi = bootstrap_ci(grp["delta"].to_numpy())
            agg_rows.append({
                "bucket": keys[0], "season": keys[1], "variable": keys[2], "pair": keys[3], "metric": keys[4],
                "n_windows": len(grp),
                "share_favorable": float(grp["favorable"].mean()),
                "delta_mean": mean, "delta_ci_lo": lo, "delta_ci_hi": hi,
            })
    pairwise_summary = pd.DataFrame(agg_rows)
    pairwise_summary.to_csv(ROBUST_ROOT / "pairwise_longlead_deltas_summary.csv", index=False)

    agg_rows = []
    if not leadwise_pairwise_by_window.empty:
        for keys, grp in leadwise_pairwise_by_window.groupby(["bucket", "season", "variable", "pair", "metric", "lead_h"], dropna=False):
            mean, lo, hi = bootstrap_ci(grp["delta"].to_numpy())
            agg_rows.append({
                "bucket": keys[0], "season": keys[1], "variable": keys[2], "pair": keys[3], "metric": keys[4], "lead_h": keys[5],
                "n_windows": len(grp),
                "share_favorable": float(grp["favorable"].mean()),
                "delta_mean": mean, "delta_ci_lo": lo, "delta_ci_hi": hi,
            })
    leadwise_pairwise_summary = pd.DataFrame(agg_rows)
    leadwise_pairwise_summary.to_csv(ROBUST_ROOT / "pairwise_leadwise_deltas_summary.csv", index=False)

    # model win share by lead/variable
    if {"rmse_mean", "acc_mean"}.issubset(work.columns):
        tmp = work.copy()
        tmp["best_rmse_model"] = tmp.groupby(["tag", "variable", "lead_h"])["rmse_mean"].transform("min") == tmp["rmse_mean"]
        tmp["best_acc_model"] = tmp.groupby(["tag", "variable", "lead_h"])["acc_mean"].transform("max") == tmp["acc_mean"]
        rows = []
        for metric, flag in [("rmse_mean", "best_rmse_model"), ("acc_mean", "best_acc_model")]:
            grp = tmp[tmp[flag]].groupby(["bucket", "season", "variable", "lead_h", "model"]).size().reset_index(name="wins")
            den = tmp.groupby(["bucket", "season", "variable", "lead_h"]).tag.nunique().reset_index(name="n_windows")
            grp = grp.merge(den, on=["bucket", "season", "variable", "lead_h"], how="left")
            grp["win_share"] = grp["wins"] / grp["n_windows"]
            grp["metric"] = metric
            rows.append(grp)
        winshare_by_lead = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
        winshare_by_lead.to_csv(ROBUST_ROOT / "model_winshare_by_lead.csv", index=False)

det_all.to_csv(ROBUST_ROOT / "all_windows_deterministic_summary.csv", index=False)
display(pairwise_summary.head(20))
display(leadwise_pairwise_summary.head(20))


,bucket,season,variable,pair,metric,n_windows,share_favorable,delta_mean,delta_ci_lo,delta_ci_hi
0,season,DJF,mslp,GraphCast_minus_HRES,acc_mean,4,0.5,0.045788,0.044830,0.046746
1,season,DJF,mslp,GraphCast_minus_HRES,rmse_mean,4,0.5,-60.707930,-66.303941,-55.111920
2,season,DJF,mslp,NeuralGCM_minus_HRES,acc_mean,4,0.0,NaN,NaN,NaN
3,season,DJF,mslp,NeuralGCM_minus_HRES,rmse_mean,4,0.0,NaN,NaN,NaN
4,season,DJF,t850,GraphCast_minus_HRES,acc_mean,4,0.5,0.047957,0.043282,0.052632
5,season,DJF,t850,GraphCast_minus_HRES,rmse_mean,4,0.5,-0.365548,-0.422852,-0.308244
6,season,DJF,t850,NeuralGCM_minus_HRES,acc_mean,4,0.5,0.026183,0.009523,0.042843
7,season,DJF,t850,NeuralGCM_minus_HRES,rmse_mean,4,0.5,-0.368174,-0.460126,-0.276222
8,season,DJF,u850,GraphCast_minus_HRES,acc_mean,4,0.5,0.079047,0.073803,0.084291
9,season,DJF,u850,GraphCast_minus_HRES,rmse_mean,4,0.5,-0.693306,-0.698799,-0.687813


,bucket,season,variable,pair,metric,lead_h,n_windows,share_favorable,delta_mean,delta_ci_lo,delta_ci_hi
0,season,DJF,mslp,GraphCast_minus_HRES,acc_mean,24.0,4,0.5,0.001668,0.001469,0.001866
1,season,DJF,mslp,GraphCast_minus_HRES,acc_mean,72.0,4,0.5,0.005605,0.004910,0.006301
2,season,DJF,mslp,GraphCast_minus_HRES,acc_mean,120.0,4,0.5,0.021250,0.020832,0.021667
3,season,DJF,mslp,GraphCast_minus_HRES,acc_mean,168.0,4,0.5,0.045788,0.044830,0.046746
4,season,DJF,mslp,GraphCast_minus_HRES,rmse_mean,24.0,4,0.5,-15.383500,-16.202524,-14.564476
5,season,DJF,mslp,GraphCast_minus_HRES,rmse_mean,72.0,4,0.5,-21.652797,-21.672260,-21.633335
6,season,DJF,mslp,GraphCast_minus_HRES,rmse_mean,120.0,4,0.5,-42.543793,-46.686592,-38.400993
7,season,DJF,mslp,GraphCast_minus_HRES,rmse_mean,168.0,4,0.5,-60.707930,-66.303941,-55.111920
8,season,DJF,mslp,NeuralGCM_minus_HRES,acc_mean,24.0,4,0.0,NaN,NaN,NaN
9,season,DJF,mslp,NeuralGCM_minus_HRES,acc_mean,72.0,4,0.0,NaN,NaN,NaN


In [6]:
# Regime balance, regime order, and entropy

regime_balance = pd.DataFrame()
regime_order = pd.DataFrame()
regime_entropy = pd.DataFrame()

if not lab_all.empty and {"tag", "regime"}.issubset(lab_all.columns):
    frac = lab_all.groupby(["tag", "bucket", "season", "regime"]).size().reset_index(name="count")
    frac["fraction"] = frac["count"] / frac.groupby("tag")["count"].transform("sum")

    regime_balance = frac.groupby(["tag", "bucket", "season"]).agg(
        min_fraction=("fraction", "min"),
        max_fraction=("fraction", "max"),
        n_regimes=("regime", "nunique"),
    ).reset_index()
    regime_balance.to_csv(ROBUST_ROOT / "regime_balance_by_window.csv", index=False)

    ent_rows = []
    for keys, grp in frac.groupby(["tag", "bucket", "season"], dropna=False):
        p = grp["fraction"].to_numpy(dtype=float)
        ent = -np.sum(np.where(p > 0, p * np.log(p), 0.0))
        max_ent = np.log(max(len(p), 1))
        ent_rows.append({
            "tag": keys[0], "bucket": keys[1], "season": keys[2],
            "regime_entropy": float(ent),
            "normalized_entropy": float(ent / max_ent) if max_ent > 0 else np.nan,
        })
    regime_entropy = pd.DataFrame(ent_rows)
    regime_entropy.to_csv(ROBUST_ROOT / "regime_entropy_by_window.csv", index=False)

if not reg_all.empty and {"tag", "variable", "metric", "model", "regime"}.issubset(reg_all.columns):
    work = reg_all.copy()
    if "lead_h" not in work.columns and "lead" in work.columns:
        work["lead_h"] = parse_lead_hours_series(work["lead"])
    work["lead_h"] = pd.to_numeric(work["lead_h"], errors="coerce")
    if "mean" not in work.columns:
        for c in ["value", "metric_mean", "avg"]:
            if c in work.columns:
                work["mean"] = work[c]
                break
    if "mean" in work.columns:
        metric_mask = work["metric"].astype(str).str.upper().eq("RMSE")
        if metric_mask.any():
            work = work[metric_mask & work["lead_h"].notna()].copy()
            work = work[work["lead_h"] == work.groupby(["tag", "model"])["lead_h"].transform("max")]
            hard = work.loc[work.groupby(["tag", "model"])["mean"].idxmax(), ["tag", "bucket", "season", "model", "regime", "mean"]].rename(columns={"regime":"hardest_regime","mean":"hardest_value"})
            easy = work.loc[work.groupby(["tag", "model"])["mean"].idxmin(), ["tag", "model", "regime", "mean"]].rename(columns={"regime":"easiest_regime","mean":"easiest_value"})
            regime_order = hard.merge(easy, on=["tag", "model"], how="left")
            regime_order.to_csv(ROBUST_ROOT / "regime_order_stability.csv", index=False)

reg_all.to_csv(ROBUST_ROOT / "all_windows_regime_metrics.csv", index=False)
display(regime_balance.head(20))
display(regime_entropy.head(20))
display(regime_order.head(20))


,tag,bucket,season,min_fraction,max_fraction,n_regimes
0,2018,year,ALL,0.158219,0.315068,4
1,2019,year,ALL,0.201370,0.349315,4
2,2020,year,ALL,0.209699,0.284153,4
3,2021,year,ALL,0.197260,0.373973,4
4,2022,year,ALL,0.183562,0.329452,4
5,DJF_2019,season,DJF,0.150000,0.319444,4
6,DJF_2020,season,DJF,0.175824,0.351648,4
7,DJF_2021,season,DJF,0.172222,0.363889,4
8,DJF_2022,season,DJF,0.161111,0.419444,4
9,JJA_2018,season,JJA,0.165761,0.317935,4


,tag,bucket,season,regime_entropy,normalized_entropy
0,2018,year,ALL,1.358143,0.979693
1,2019,year,ALL,1.360953,0.981720
2,2020,year,ALL,1.380585,0.995881
3,2021,year,ALL,1.348185,0.972510
4,2022,year,ALL,1.364710,0.984430
5,DJF_2019,season,DJF,1.350515,0.974191
6,DJF_2020,season,DJF,1.353345,0.976232
7,DJF_2021,season,DJF,1.335885,0.963637
8,DJF_2022,season,DJF,1.313488,0.947481
9,JJA_2018,season,JJA,1.361414,0.982052


,tag,bucket,season,model,hardest_regime,hardest_value,easiest_regime,easiest_value
0,2018,year,ALL,HRES,2,545.738149,0,2.629897
1,2019,year,ALL,GraphCast,1,442.962716,1,2.195571
2,2019,year,ALL,HRES,0,540.153905,2,2.603071
3,2020,year,ALL,GraphCast,1,489.064821,2,2.276802
4,2020,year,ALL,HRES,1,539.559435,2,2.562609
5,2020,year,ALL,NeuralGCM,1,487.996550,2,2.286790
6,2021,year,ALL,GraphCast,3,449.397582,3,2.283104
7,2021,year,ALL,HRES,2,536.450178,2,2.691776
8,2022,year,ALL,HRES,1,541.193252,0,2.587846
9,DJF_2019,season,DJF,HRES,1,530.566963,2,2.492357


In [7]:
# Blocking penalty and blocking event skill summaries

blocking_penalty = pd.DataFrame()
blocking_penalty_summary = pd.DataFrame()
blocking_event_skill = pd.DataFrame()

if not blk_all.empty and {"tag", "bucket", "season", "model"}.issubset(blk_all.columns):
    work = blk_all.copy()
    if "lead_h" not in work.columns and "lead" in work.columns:
        work["lead_h"] = parse_lead_hours_series(work["lead"])
    work["lead_h"] = pd.to_numeric(work["lead_h"], errors="coerce")
    if {"blocked_rmse", "unblocked_rmse"}.issubset(work.columns):
        if work["lead_h"].notna().any():
            work = work[work["lead_h"] == work.groupby(["tag", "model"])["lead_h"].transform("max")].copy()
        work["blocking_penalty"] = work["blocked_rmse"] - work["unblocked_rmse"]
        blocking_penalty = work.copy()
        blocking_penalty.to_csv(ROBUST_ROOT / "blocking_penalty_by_window.csv", index=False)

        rows = []
        for keys, grp in blocking_penalty.groupby(["bucket", "season", "model"], dropna=False):
            mean, lo, hi = bootstrap_ci(grp["blocking_penalty"].to_numpy())
            rows.append({
                "bucket": keys[0], "season": keys[1], "model": keys[2],
                "n_windows": len(grp),
                "share_positive_penalty": float((grp["blocking_penalty"] > 0).mean()),
                "penalty_mean": mean, "penalty_ci_lo": lo, "penalty_ci_hi": hi,
            })
        blocking_penalty_summary = pd.DataFrame(rows)
        blocking_penalty_summary.to_csv(ROBUST_ROOT / "blocking_penalty_summary.csv", index=False)

if not evt_all.empty and {"bucket", "season", "model"}.issubset(evt_all.columns):
    work = evt_all.copy()
    if "lead_h" not in work.columns and "lead" in work.columns:
        work["lead_h"] = parse_lead_hours_series(work["lead"])
    work["lead_h"] = pd.to_numeric(work["lead_h"], errors="coerce")
    if work["lead_h"].notna().any():
        work = work[work["lead_h"] == work.groupby(["tag", "model"])["lead_h"].transform("max")].copy()

    metric_candidates = [c for c in ["pod", "far", "csi", "frequency_bias", "bias", "hits", "misses", "false_alarms", "correct_negatives"] if c in work.columns]
    rows = []
    for keys, grp in work.groupby(["bucket", "season", "model"], dropna=False):
        row = {"bucket": keys[0], "season": keys[1], "model": keys[2], "n_windows": len(grp)}
        for c in metric_candidates:
            mean, lo, hi = bootstrap_ci(pd.to_numeric(grp[c], errors="coerce").to_numpy())
            row[f"{c}_mean"] = mean
            row[f"{c}_ci_lo"] = lo
            row[f"{c}_ci_hi"] = hi
        rows.append(row)
    blocking_event_skill = pd.DataFrame(rows)
    blocking_event_skill.to_csv(ROBUST_ROOT / "blocking_event_skill_summary.csv", index=False)

blk_all.to_csv(ROBUST_ROOT / "all_windows_blocking_rmse.csv", index=False)
evt_all.to_csv(ROBUST_ROOT / "all_windows_blocking_event_metrics.csv", index=False)
display(blocking_penalty_summary.head(20))
display(blocking_event_skill.head(20))


,bucket,season,model,n_windows,share_positive_penalty,penalty_mean,penalty_ci_lo,penalty_ci_hi
0,season,DJF,GraphCast,4,0.50,12.931088,10.942685,14.919492
1,season,DJF,HRES,4,0.75,17.859589,4.777164,29.424085
2,season,DJF,NeuralGCM,4,0.25,12.177938,-15.512877,39.868753
3,season,JJA,GraphCast,5,0.20,26.306148,26.306148,26.306148
4,season,JJA,HRES,5,1.00,13.814808,4.969083,25.044311
5,season,JJA,NeuralGCM,5,0.20,13.576145,13.576145,13.576145
6,season,MAM,GraphCast,5,0.20,38.573073,38.573073,38.573073
7,season,MAM,HRES,5,0.40,-2.156357,-9.377401,4.769438
8,season,MAM,NeuralGCM,5,0.20,17.754705,17.754705,17.754705
9,season,SON,GraphCast,5,0.20,-8.167624,-35.629951,19.294702


,bucket,season,model,n_windows,frequency_bias_mean,frequency_bias_ci_lo,frequency_bias_ci_hi,hits_mean,hits_ci_lo,hits_ci_hi,misses_mean,misses_ci_lo,misses_ci_hi,false_alarms_mean,false_alarms_ci_lo,false_alarms_ci_hi,correct_negatives_mean,correct_negatives_ci_lo,correct_negatives_ci_hi
0,season,DJF,GraphCast,8,0.564426,0.176471,1.064426,4.000,0.7500,8.6250,2.625,0.3750,5.8750,1.875,0.3750,3.6250,64.50,18.85625,111.000000
1,season,DJF,HRES,8,0.993330,0.808736,1.209233,11.375,7.0000,16.0000,8.625,4.8750,12.7500,9.250,6.0000,12.7500,137.25,129.12500,146.253125
2,season,DJF,NeuralGCM,8,0.381250,0.104167,0.750000,3.875,0.7500,8.3750,1.625,0.3750,3.3750,1.750,0.5000,3.5000,34.75,11.00000,62.000000
3,season,JJA,GraphCast,10,0.182941,0.000000,0.468824,2.300,0.0000,6.8000,1.600,0.0000,4.4000,1.600,0.0000,4.5000,28.50,0.00000,69.300000
4,season,JJA,HRES,10,0.937690,0.701150,1.246318,12.800,7.3975,17.8025,14.400,8.5975,21.5025,8.700,6.5000,11.0025,134.10,122.40000,145.802500
5,season,JJA,NeuralGCM,10,0.291176,0.000000,0.782353,2.400,0.0000,7.1000,1.500,0.0000,4.1000,1.700,0.0000,4.2000,28.40,0.00000,69.600000
6,season,MAM,GraphCast,10,0.134857,0.000000,0.340681,4.800,0.0000,13.4000,5.000,0.0000,12.4000,1.900,0.0000,5.1000,22.30,0.00000,54.800000
7,season,MAM,HRES,10,1.032852,0.910192,1.200256,30.900,20.5000,41.9000,14.800,10.9000,18.5025,13.600,11.2975,15.8000,110.70,100.50000,121.005000
8,season,MAM,NeuralGCM,10,0.177330,0.000000,0.451434,6.500,0.0000,17.8000,3.300,0.0000,8.0000,2.400,0.0000,6.0000,21.80,0.00000,53.200000
9,season,SON,GraphCast,10,0.271712,0.060000,0.518378,2.500,0.0000,7.0000,2.500,0.3000,5.7000,1.100,0.1000,2.6000,30.70,1.00000,71.900000


In [8]:
# Dense-lead manifest and final exports

dense_summary_path = DENSE_ROOT / "denselead_summary.json"
dense_summary = {}
if dense_summary_path.exists():
    dense_summary = json.loads(dense_summary_path.read_text(encoding="utf-8"))

dense_df = pd.DataFrame([dense_summary]) if dense_summary else pd.DataFrame()
dense_df.to_csv(ROBUST_ROOT / "denselead_manifest_summary.csv", index=False)

# quick integrity summary
summary_rows = [{
    "eligible_windows": len(manifest),
    "det_rows": len(det_all),
    "reg_rows": len(reg_all),
    "blk_rows": len(blk_all),
    "evt_rows": len(evt_all),
    "label_rows": len(lab_all),
    "have_pairwise_summary": int(not pairwise_summary.empty),
    "have_leadwise_pairwise_summary": int(not leadwise_pairwise_summary.empty),
    "have_regime_entropy": int(not regime_entropy.empty),
    "have_blocking_event_skill": int(not blocking_event_skill.empty),
    "have_dense_summary": int(not dense_df.empty),
}]
integrity = pd.DataFrame(summary_rows)
integrity.to_csv(ROBUST_ROOT / "robustness_integrity_summary.csv", index=False)

display(integrity)
display(coverage)
display(pairwise_summary.head(20))
display(blocking_penalty_summary.head(20))
display(dense_df)


,eligible_windows,det_rows,reg_rows,blk_rows,evt_rows,label_rows,have_pairwise_summary,have_leadwise_pairwise_summary,have_regime_entropy,have_blocking_event_skill,have_dense_summary
0,24,1344,11224,288,576,14248,1,1,1,1,1


,bucket,season,n_windows
0,season,DJF,4
1,season,JJA,5
2,season,MAM,5
3,season,SON,5
4,year,ALL,5


,bucket,season,variable,pair,metric,n_windows,share_favorable,delta_mean,delta_ci_lo,delta_ci_hi
0,season,DJF,mslp,GraphCast_minus_HRES,acc_mean,4,0.5,0.045788,0.044830,0.046746
1,season,DJF,mslp,GraphCast_minus_HRES,rmse_mean,4,0.5,-60.707930,-66.303941,-55.111920
2,season,DJF,mslp,NeuralGCM_minus_HRES,acc_mean,4,0.0,NaN,NaN,NaN
3,season,DJF,mslp,NeuralGCM_minus_HRES,rmse_mean,4,0.0,NaN,NaN,NaN
4,season,DJF,t850,GraphCast_minus_HRES,acc_mean,4,0.5,0.047957,0.043282,0.052632
5,season,DJF,t850,GraphCast_minus_HRES,rmse_mean,4,0.5,-0.365548,-0.422852,-0.308244
6,season,DJF,t850,NeuralGCM_minus_HRES,acc_mean,4,0.5,0.026183,0.009523,0.042843
7,season,DJF,t850,NeuralGCM_minus_HRES,rmse_mean,4,0.5,-0.368174,-0.460126,-0.276222
8,season,DJF,u850,GraphCast_minus_HRES,acc_mean,4,0.5,0.079047,0.073803,0.084291
9,season,DJF,u850,GraphCast_minus_HRES,rmse_mean,4,0.5,-0.693306,-0.698799,-0.687813


,bucket,season,model,n_windows,share_positive_penalty,penalty_mean,penalty_ci_lo,penalty_ci_hi
0,season,DJF,GraphCast,4,0.50,12.931088,10.942685,14.919492
1,season,DJF,HRES,4,0.75,17.859589,4.777164,29.424085
2,season,DJF,NeuralGCM,4,0.25,12.177938,-15.512877,39.868753
3,season,JJA,GraphCast,5,0.20,26.306148,26.306148,26.306148
4,season,JJA,HRES,5,1.00,13.814808,4.969083,25.044311
5,season,JJA,NeuralGCM,5,0.20,13.576145,13.576145,13.576145
6,season,MAM,GraphCast,5,0.20,38.573073,38.573073,38.573073
7,season,MAM,HRES,5,0.40,-2.156357,-9.377401,4.769438
8,season,MAM,NeuralGCM,5,0.20,17.754705,17.754705,17.754705
9,season,SON,GraphCast,5,0.20,-8.167624,-35.629951,19.294702


,all_years_tag,requested_candidate_leads_h,selected_dense_leads_h,n_selected_dense_leads
0,all_2018_2022,"[24, 48, 72, 96, 120, 144, 168]","[24, 48, 72, 96, 120, 144, 168]",7
